# Chapter 2 Practical 04: Evaluation of Top-K Recommendations

Learning objectives:
- Compute Precision@K, Recall@K, and HitRate@K.
- Evaluate recommendations for multiple users.
- Read a compact metric table and bar chart.
- Distinguish accuracy, coverage, and hit-oriented satisfaction.

Slide connection: Precision@K, Recall@K, HitRate@K, ranking, and Top-N evaluation.


We use a tiny ground-truth example. In a real project, relevant items usually come from held-out ratings, clicks, purchases, or watch events.


In [1]:
# Teaching note: Load movies and interactions, then define the held-out relevant items.
import pandas as pd
import matplotlib.pyplot as plt

ground_truth = {
    "U1": {"The Martian", "Gravity", "The Matrix"},
    "U2": {"Toy Story", "Finding Nemo", "Paddington"},
    "U3": {"Titanic", "The Notebook", "La La Land"},
}

recommendations = {
    "U1": ["The Martian", "The Dark Knight", "Gravity", "Titanic", "Paddington"],
    "U2": ["Paddington", "Toy Story", "The Martian", "Finding Nemo", "Inception"],
    "U3": ["La La Land", "Titanic", "Interstellar", "The Notebook", "Toy Story"],
}

pd.DataFrame([
    {"user": u, "ground_truth": sorted(gt), "recommendations": recommendations[u]}
    for u, gt in ground_truth.items()
])


,user,ground_truth,recommendations
0,U1,"[Gravity, The Martian, The Matrix]","[The Martian, The Dark Knight, Gravity, Titani..."
1,U2,"[Finding Nemo, Paddington, Toy Story]","[Paddington, Toy Story, The Martian, Finding N..."
2,U3,"[La La Land, The Notebook, Titanic]","[La La Land, Titanic, Interstellar, The Notebo..."


Precision@K asks: of the K recommended items, how many were relevant?


In [2]:
# Teaching note: Create a tiny recommendation list so Top-K metrics can be computed by hand.
# Precision@K answers: of the K recommended items, how many were relevant?
def precision_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / k

# Recall@K answers: of all relevant items, how many did the recommender find?
def recall_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / len(relevant) if relevant else 0

def hitrate_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    return int(len(set(recommended_k) & set(relevant)) > 0)

precision_at_k(recommendations["U1"], ground_truth["U1"], k=3)


0.6666666666666666

Now evaluate all users and display the results in one clean table.


In [3]:
# Teaching note: Define Precision@K, Recall@K, and HitRate@K for recommendation evaluation.
k = 3
rows = []
for user, relevant in ground_truth.items():
    recs = recommendations[user]
    rows.append({
        "user": user,
        f"Precision@{k}": precision_at_k(recs, relevant, k),
        f"Recall@{k}": recall_at_k(recs, relevant, k),
        f"HitRate@{k}": hitrate_at_k(recs, relevant, k),
        "hits_in_top_k": sorted(set(recs[:k]) & relevant),
    })

metrics = pd.DataFrame(rows)
metrics


,user,Precision@3,Recall@3,HitRate@3,hits_in_top_k
0,U1,0.666667,0.666667,1,"[Gravity, The Martian]"
1,U2,0.666667,0.666667,1,"[Paddington, Toy Story]"
2,U3,0.666667,0.666667,1,"[La La Land, Titanic]"


Average the metrics across users to summarize system performance.


In [4]:
# Teaching note: Evaluate the recommendation list at multiple K values.
summary = metrics[[f"Precision@{k}", f"Recall@{k}", f"HitRate@{k}"]].mean().to_frame("mean_value")
summary.round(3)


,mean_value
Precision@3,0.667
Recall@3,0.667
HitRate@3,1.000


A simple bar chart makes it easier to compare the metric values.


In [5]:
# Teaching note: Plot metric values so students can see how K changes evaluation behavior.
ax = summary.plot(kind="bar", legend=False, ylim=(0, 1), figsize=(6, 4))
ax.set_ylabel("Metric value")
ax.set_title(f"Average Top-{k} evaluation")
ax.bar_label(ax.containers[0], fmt="%.2f")
plt.xticks(rotation=0)
plt.show()


/var/folders/w0/2jgnn0bx27b_jyy4mrm45cxm0000gn/T/ipykernel_51798/1570277064.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Accuracy, coverage, and satisfaction are related but not identical:

- Accuracy asks whether recommended items match known relevant items.
- Coverage asks whether the system can recommend a broad set of items instead of always the same few.
- HitRate@K asks whether the list contains at least one useful item for the user.

## What did we learn?

- Precision@K rewards short lists with many relevant items.
- Recall@K rewards finding a large share of all relevant items.
- HitRate@K is forgiving: one hit is enough.

Exercises:
1. Change `k` from 3 to 5. Which metric changes the most?
2. Add one more user with recommendations and ground truth.
